# Kubeflow Pipeline Definitions
Defines and compiles all Kubeflow pipelines for the code-understanding workflows.
Run via `run_pipelines.sh` with `PIPELINE_COMPILE_ONLY=1` to compile all pipelines to YAML.

In [ ]:
##############################################################################
# Imports
##############################################################################
import os
from kfp import dsl
from kfp.dsl import Artifact, Dataset, Output
from utils.pipeline_utils import BASE_IMAGE, DATA_GENERATION_BASE_IMAGE, INDEXING_BASE_IMAGE, ANALYSIS_BASE_IMAGE, compile_all_and_exit

## Single-Repo Data Generation Pipeline
Clones one git repository, detects languages, and generates GraphRAG code metadata.

In [ ]:
##############################################################################
# Single-Repo Data Generation Pipeline
##############################################################################

@dsl.component(base_image=DATA_GENERATION_BASE_IMAGE)
def prepare_environment_op(git_repo: str, git_branch: str, source_path: str, target_path: str):
    import os, sys

    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

    from pipelines.data_generation import prepare_environment

    prepare_environment(source_path=source_path, target_path=target_path,
                        git_repo=git_repo, git_branch=git_branch)


@dsl.component(base_image=DATA_GENERATION_BASE_IMAGE)
def generate_code_and_meta_op(git_repo: str, git_branch: str, source_path: str, target_path: str):
    import os, sys

    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

    from pipelines.data_generation import generate_all_code_and_meta

    generate_all_code_and_meta(git_repo=git_repo, git_branch=git_branch,
                               source_path=source_path, target_path=target_path)


@dsl.pipeline(name="single-data-generation-pipeline")
def single_data_generation_pipeline(
    git_repo: str = os.getenv("GIT_REPO", ""),
    git_branch: str = os.getenv("GIT_BRANCH", "main"),
    source_path: str = os.getenv("SOURCE_PATH", "source"),
    target_path: str = os.getenv("TARGET_PATH", "target"),
):
    prep = prepare_environment_op(
        git_repo=git_repo, git_branch=git_branch,
        source_path=source_path, target_path=target_path,
    )

    generate_code_and_meta_op(
        git_repo=git_repo, git_branch=git_branch,
        source_path=source_path, target_path=target_path,
    ).after(prep)

## Aggregated Data Generation Pipeline
Processes multiple git repositories in parallel, generating GraphRAG metadata for each.

In [ ]:
##############################################################################
# Aggregated Data Generation Pipeline
##############################################################################

_GIT_REPOS = [
    {"git_repo": "https://github.com/agapebondservant/tic-tac-toe-sample",   "git_branch": "main",                     "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://github.com/agapebondservant/spacefx-sample-javafx", "git_branch": "main",                     "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://github.com/asciimoo/yappb",                         "git_branch": "master",                   "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://github.com/ParthKolekar/pacman-game",               "git_branch": "master",                   "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://gitlab.com/agapebondservant/ticket-monster",        "git_branch": "2.7.0.Final-with-tutorials", "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://github.com/Sajith-Dilshan/Authentication-Module-Java-Web-Project", "git_branch": "master",    "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://github.com/banago/simple-php-website",              "git_branch": "main",                     "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://github.com/agapebondservant/lamp-simple-rhel7",     "git_branch": "main",                     "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://github.com/jariolaine/apex-blog",                   "git_branch": "master",                   "parent_source_path": "source", "parent_target_path": "target"},
    {"git_repo": "https://github.com/Tzoop/MemoryGame",                       "git_branch": "master",                   "parent_source_path": "source", "parent_target_path": "target"},
]


@dsl.component(base_image=BASE_IMAGE)
def run_repo_full_op(
    git_repo: str,
    git_branch: str,
    parent_source_path: str,
    parent_target_path: str,
    graphrag_base_path: str,
    adhoc_question: str,
    result: Output[Dataset],
):
    import os, sys, json

    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

    from utils import code_utils
    from pipelines.data_generation import prepare_environment, generate_all_code_and_meta
    from pipelines.indexing import run_full_pipeline as run_indexing
    from pipelines.analysis import run_full_pipeline as run_analysis

    repo_slug = code_utils.generate_slug_from_repo(git_repo, git_branch)

    source_path = f"{parent_source_path}/{repo_slug}"

    target_path = f"{parent_target_path}/{repo_slug}"

    graphrag_source_path = f"{graphrag_base_path}/{repo_slug}"

    outcome = {"git_slug": repo_slug, "status": "fail", "fail_message": ""}

    try:

        prepare_environment(source_path=source_path, target_path=target_path,
                            git_repo=git_repo, git_branch=git_branch)

        generate_all_code_and_meta(git_repo=git_repo, git_branch=git_branch,
                                   source_path=source_path, target_path=target_path,
                                   git_slug=repo_slug, multi_repo=True)

        run_indexing(codebase_path=target_path, graphrag_source_path=graphrag_source_path,
                     git_slug=repo_slug, multi_repo=True)

        run_analysis(graphrag_source_path=graphrag_source_path)

        outcome["status"] = "success"

    except Exception as e:

        outcome["fail_message"] = str(e)

        import logging
        logging.error(f"Error processing '{git_repo}': {e}")

    with open(result.path, "w") as f:
        json.dump(outcome, f)


@dsl.pipeline(name="aggregated-data-generation-pipeline")
def aggregated_data_generation_pipeline(
    parent_source_path: str = "source",
    parent_target_path: str = "target",
    graphrag_base_path: str = "graph_rag_app/source",
    adhoc_question: str = "Which modules or components would be riskiest to refactor first? Include the fully qualified names.",
):
    with dsl.ParallelFor(items=_GIT_REPOS) as repo:

        run_repo_full_op(
            git_repo=repo["git_repo"],
            git_branch=repo["git_branch"],
            parent_source_path=parent_source_path,
            parent_target_path=parent_target_path,
            graphrag_base_path=graphrag_base_path,
            adhoc_question=adhoc_question,
        )

## GraphRAG Indexing Pipeline
Builds a GraphRAG index from the generated code metadata and stores it in LanceDB.

In [ ]:
##############################################################################
# GraphRAG Indexing Pipeline
##############################################################################

@dsl.component(base_image=INDEXING_BASE_IMAGE)
def graphrag_indexing_op(
    codebase_path: str,
    graphrag_source_path: str,
    result: Output[Artifact],
):
    import os, sys, json

    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

    from pipelines.indexing import run_full_pipeline

    pipeline_result = run_full_pipeline(codebase_path=codebase_path,
                                        graphrag_source_path=graphrag_source_path)

    with open(result.path, "w") as f:
        json.dump(pipeline_result, f)

    if pipeline_result.get("status") != "success":
        raise RuntimeError(f"GraphRAG indexing failed: {pipeline_result.get('fail_message')}")


@dsl.pipeline(name="graphrag-indexing-pipeline")
def graphrag_indexing_pipeline(
    codebase_path: str = "target",
    graphrag_source_path: str = "graph_rag_app/source",
):
    graphrag_indexing_op(
        codebase_path=codebase_path,
        graphrag_source_path=graphrag_source_path,
    )

## GraphRAG Analysis Pipeline
Generates a migration report and runs an ad-hoc query against the GraphRAG index.

In [ ]:
##############################################################################
# GraphRAG Analysis Pipeline
##############################################################################

@dsl.component(base_image=ANALYSIS_BASE_IMAGE)
def generate_migration_report_op(graphrag_source_path: str, report: Output[Artifact]):
    import os, sys

    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

    from pipelines.analysis import run_full_pipeline

    migration_report = run_full_pipeline(graphrag_source_path)

    with open(report.path, "w") as f:
        f.write(migration_report)


@dsl.component(base_image=ANALYSIS_BASE_IMAGE)
def run_adhoc_query_op(graphrag_source_path: str, question: str, result: Output[Artifact]):
    import asyncio, logging

    logging.basicConfig(level=logging.INFO)

    from utils.graphrag_utils import DependencyAnalyzer

    analyzer = DependencyAnalyzer(graphrag_source_path)

    response = asyncio.run(analyzer.query_with_llm(question))

    with open(result.path, "w") as f:
        f.write(response)


@dsl.pipeline(name="graphrag-analysis-pipeline")
def graphrag_analysis_pipeline(
    graphrag_source_path: str = "graph_rag_app/source",
    adhoc_question: str = "Which modules or components would be riskiest to refactor first? Include the fully qualified names.",
):
    migration = generate_migration_report_op(graphrag_source_path=graphrag_source_path)

    run_adhoc_query_op(
        graphrag_source_path=graphrag_source_path,
        question=adhoc_question,
    ).after(migration)

## Compile & Submit
When `PIPELINE_COMPILE_ONLY=1`, compiles all pipelines to `$KFP_PIPELINE_OUTPUT_DIR/<name>.yaml` and exits.

## Single-Repo Full Pipeline
Chains data generation → GraphRAG indexing → analysis in a single ordered pipeline.
Each stage waits for the previous to succeed before starting.

In [ ]:
##############################################################################
# Single-Repo Full Pipeline (Data Generation → Indexing → Analysis)
##############################################################################

@dsl.pipeline(name="single-full-pipeline")
def single_full_pipeline(
    git_repo: str = os.getenv("GIT_REPO", ""),
    git_branch: str = os.getenv("GIT_BRANCH", "main"),
    source_path: str = os.getenv("SOURCE_PATH", "source"),
    target_path: str = os.getenv("TARGET_PATH", "target"),
    graphrag_source_path: str = "graph_rag_app/source",
    adhoc_question: str = "Which modules or components would be riskiest to refactor first? Include the fully qualified names.",
):
    prep = prepare_environment_op(
        git_repo=git_repo, git_branch=git_branch,
        source_path=source_path, target_path=target_path,
    )

    gen = generate_code_and_meta_op(
        git_repo=git_repo, git_branch=git_branch,
        source_path=source_path, target_path=target_path,
    ).after(prep)

    idx = graphrag_indexing_op(
        codebase_path=target_path,
        graphrag_source_path=graphrag_source_path,
    ).after(gen)

    migration = generate_migration_report_op(
        graphrag_source_path=graphrag_source_path,
    ).after(idx)

    run_adhoc_query_op(
        graphrag_source_path=graphrag_source_path,
        question=adhoc_question,
    ).after(migration)

In [ ]:
##############################################################################
# Compile all pipelines
##############################################################################
compile_all_and_exit({
    "single":      single_data_generation_pipeline,
    "single_full": single_full_pipeline,
    "aggregated":  aggregated_data_generation_pipeline,
    "indexing":    graphrag_indexing_pipeline,
    "analysis":    graphrag_analysis_pipeline,
})